In [1]:
#### IMPORTS
import numpy as np

In [4]:
#### UTILITIES 
def create_adjacancy__matrix(block_sizes):
    """
    Create a block matrix with ones on the diagonal blocks and zeros elsewhere.
    
    Parameters:
    block_sizes (list of int): Sizes of the blocks along the diagonal.
    
    Returns:
    np.ndarray: The resulting block matrix.
    """

    K = len(block_sizes)
    # Calculate the total size of the matrix
    total_size = sum(block_sizes)
    # Initialize the full matrix with zeros
    block_matrix = np.zeros((total_size, total_size), dtype=int)
    
    # Fill the diagonal blocks with ones
    start_index = 0
    for size in block_sizes:
        block_matrix[start_index:start_index + size, start_index:start_index + size] = 1
        start_index += size
    
    eigenvalues, eigenvectors = np.linalg.eigh(block_matrix)
    top_indices = np.argsort(eigenvalues)[-K:]  # Indices of the K largest eigenvalues
    Lambda = eigenvalues[top_indices]   # The K largest eigenvalues
    E = eigenvectors[:, top_indices] # The eigenvectors corresponding to the largest eigenvalues, the dimension is nxK 

    return block_matrix, Lambda, E

def create_Ri_symmetric_matrix_with_constrained_eigenvalues(dimension, rho_minus, rho_plus, noise_level=0.0, include_extremes=True):
    """
    Create a symmetric matrix with eigenvalues between rho_minus and rho_plus.
    
    Parameters:
    dimension (int): Size of the matrix (n x n).
    rho_minus (float): Lower bound of the eigenvalues.
    rho_plus  (float): Upper bound of the eigenvalues.
    noise_level (float): The scale of the noise to be added (small value to slightly perturb).
    include_extremes (bool): If True, include rho_minus and rho_plus as exact eigenvalues.

    Returns:
    np.ndarray: The symmetric matrix with eigenvalues in the range [rho_minus, rho_plus].
    """
    if include_extremes:
        if dimension < 2:
            raise ValueError("Dimension must be at least 2 to include both rho_minus and rho_plus as eigenvalues.")
        # Generate the remaining dimension - 2 eigenvalues uniformly between rho_minus and rho_plus
        additional_eigenvalues = np.random.uniform(rho_minus, rho_plus, dimension - 2)
        # Combine all eigenvalues
        all_eigenvalues = np.concatenate(([rho_minus, rho_plus], additional_eigenvalues))
    else:
        # Generate dimension eigenvalues uniformly distributed between rho_minus and rho_plus
        all_eigenvalues = np.random.uniform(rho_minus, rho_plus, dimension)
    
    # Create rho_minus random orthogonal matrix Q using QR decomposition
    Q, _ = np.linalg.qr(np.random.randn(dimension, dimension))

    # Create symmetric noise matrix
    noise_matrix = noise_level*np.random.randn(dimension,dimension)
    symmetric_noise = (noise_matrix + noise_matrix.T) / 2

    # Construct the symmetric matrix using A = Q * Lambda * Q.T
    Lambda = np.diag(all_eigenvalues)  # Diagonal matrix of eigenvalues
    symmetric_matrix = Q @ Lambda @ Q.T + symmetric_noise
    
    return symmetric_matrix

def create_multiple_noisy_probability_vectors(count, dimension, noise_level=0.0, manual_probabilities=None):
    """
    Create multiple noisy probability vectors where each starts with a unique index set to 1.
    
    Parameters:
    count (int): Number of probability vectors to generate.(In our problem: the number of clusters)
    dimension (int): Length of each probability vector.(In our probelm: the number of classes)
    noise_level (float): The scale of the noise to add.
    manual_probabilities (list of lists or None): If provided, use this as the probability vectors.

    Returns:
    np.ndarray: An array containing the noisy probability vectors.
    """

    # If manual probabilities are provided, return them as a numpy array
    if manual_probabilities is not None:
        # Check that manual probabilities have the correct shape
        manual_probabilities = np.array(manual_probabilities)
        if manual_probabilities.shape != (count, dimension):
            raise ValueError(f"manual_probabilities must have shape ({count}, {dimension})")
        
        # Ensure that each row sums to 1 to confirm they are valid probability vectors
        if not np.allclose(np.sum(manual_probabilities, axis=1), 1):
            raise ValueError("Each probability vector must sum to 1")
        
        return manual_probabilities
    
    # Check that count does not exceed dimension
    if count > dimension:
        raise ValueError("The number of vectors 'count' cannot exceed the dimension size.")
    
    probability_vectors = []
    
    # Step 1: Create 'count' one-hot vectors at unique indices
    for i in range(count):
        # Initialize a one-hot vector with 1 at index i
        probability_vector = np.zeros(dimension)
        probability_vector[i] = 1
        
        # Step 2: Add random noise to the vector
        noise = np.random.uniform(0, noise_level, dimension)
        noisy_vector = probability_vector + noise
        
        # Step 3: Normalize to make it a probability vector
        noisy_probability_vector = noisy_vector / np.sum(noisy_vector)
        
        probability_vectors.append(noisy_probability_vector)
    
    return np.array(probability_vectors)

In [8]:
#### Parameters
samples_per_cluster = [4, 3, 2]             # Number of training samples in each cluster
feature_dim = 5                             # Dimension of the features at each node
rho_min = 2                                 # Minimum eigenvalue of Ri
rho_max = 3                                 # Maximum eigenvalue of Ri
noise_level_Ri = 0.0                        # Scale of noise to adjust eigenvalues
include_rhos = True                         # Whether rho_min and rho_max are exact eigenvalues
num_clusters = len(samples_per_cluster)     # Number of clusters for clustering 
num_classes = num_clusters + 0              # Number of classes for classification

#### Create the graph and the data based on our model expressions

## Set up an adjacency matrix with num_clusters perfect clusters
A, eigenvalues, E = create_adjacancy__matrix(samples_per_cluster)

#### write the code for other graphs 
#### ???????????????

## Generate symmetric matrices Ri
Ri_matrices = np.stack([create_Ri_symmetric_matrix_with_constrained_eigenvalues(feature_dim, rho_min, rho_max, noise_level=noise_level_Ri, include_extremes=include_rhos)
    for _ in range(num_clusters)], axis=0)

## Calculate Rv based on the provided expression
Rv_matrices = np.einsum("ij,jkn->ikn", ((E**2) @ np.diag(eigenvalues)), Ri_matrices)

## Generate feature vectors X based on the multivariate normal distribution
X = np.array([np.random.multivariate_normal(mean=np.zeros(feature_dim), cov=Rv, size=1).flatten() 
    for Rv in Rv_matrices])

## Generate the probability vector matrix b and generate matrix pi based on the provided expression
b = create_multiple_noisy_probability_vectors(num_clusters, num_classes, noise_level=0.0)
pi = ((E**2) @ np.diag(eigenvalues)) @ b

## Labels y based on the probability distributions in pi
y = np.array([np.random.choice(num_classes, p=pi_row) for pi_row in pi])

## Output for verification
# print("E Matrix:\n", E)
# print("\nPi Matrix:\n", pi)